## Прогноз виручки (three-phase linear)

Зошит повторює ключові кроки `3p_linear_model`: базовий Holt-Winters, сезонні та лагові ознаки, фінальна модель XGBoost для кожної товарної групи.

In [4]:
from pathlib import Path

import numpy as np
import pandas as pd
%pip install xgboost
from three_phase_linear import ForecastConfig, run_three_phase_forecast

DATA_PATH = Path('forecast_of_market_dataset.csv')
OUTPUT_PATH = Path('money_three_phase_forecast.csv')
GROUP_COLS = ['category_id']
TARGET_COLUMN = 'revenue'
REGRESSORS = ['is_sale_prohibition', 'cos_month', 'sin_month', 'cos_quarter', 'sin_quarter', 'unique_brand_count']


  Using cached xgboost-3.2.0-py3-none-win_amd64.whl.metadata (2.1 kB)
Using cached xgboost-3.2.0-py3-none-win_amd64.whl (101.7 MB)


In [9]:
# try to read with semicolon first, if that yields a single column, try comma
df = pd.read_csv(DATA_PATH, sep=';', engine='python')
if df.shape[1] == 1:
    df = pd.read_csv(DATA_PATH, sep=',', engine='python')

# remove accidental spaces from column names
df.columns = df.columns.str.strip()

# find and normalize date-like column (accept 'date' or 'month' and similar)
cols_lc = df.columns.str.lower()
date_candidates = ['date', 'month', 'day', 'dt', 'time', 'period']
for cand in date_candidates:
    if cand in cols_lc:
        orig = df.columns[cols_lc.tolist().index(cand)]
        df['date'] = pd.to_datetime(df[orig], dayfirst=True, errors='coerce')
        break
else:
    raise KeyError("No column named 'date' (or alternative) found; columns: " + ", ".join(df.columns))

# map common column name variants to the notebook's expected names
if 'category_id' not in df.columns:
    if 'product_group_id' in cols_lc:
        orig = df.columns[cols_lc.tolist().index('product_group_id')]
        df['category_id'] = pd.to_numeric(df[orig], errors='coerce')

if 'category_title' not in df.columns:
    if 'product_group_name' in cols_lc:
        orig = df.columns[cols_lc.tolist().index('product_group_name')]
        df['category_title'] = df[orig].astype(str)

# ensure a revenue column exists (prefer TARGET_COLUMN, else pick a column containing 'revenue')
if TARGET_COLUMN not in df.columns:
    revenue_matches = [c for c in df.columns if 'revenue' in c.lower()]
    if revenue_matches:
        df[TARGET_COLUMN] = pd.to_numeric(df[revenue_matches[0]], errors='coerce')
    else:
        # create an empty revenue column if nothing found
        df[TARGET_COLUMN] = np.nan

# make sure all regressors exist so subsequent conversion won't fail
for col in REGRESSORS:
    if col not in df.columns:
        df[col] = np.nan

df = df.sort_values(GROUP_COLS + ['date']).reset_index(drop=True)
for col in REGRESSORS:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# aggregate to the expected shape
agg_df = df.groupby(['date', 'category_id', 'category_title'], as_index=False).agg(
    revenue_sum=(TARGET_COLUMN, 'sum'),
    revenue_count=(TARGET_COLUMN, 'count'),
    is_sale_prohibition=('is_sale_prohibition', 'max'),
    cos_month=('cos_month', 'mean'),
    sin_month=('sin_month', 'mean'),
    cos_quarter=('cos_quarter', 'mean'),
    sin_quarter=('sin_quarter', 'mean'),
    unique_brand_count=('unique_brand_count', 'mean'),
)
agg_df.loc[agg_df['revenue_count'] == 0, 'revenue_sum'] = np.nan
df = agg_df.rename(columns={'revenue_sum': 'revenue'}).drop(columns=['revenue_count'])
df = df.sort_values(GROUP_COLS + ['date']).reset_index(drop=True)

print(df.columns.tolist())
print(df.head())

future_counts = df[df[TARGET_COLUMN].isna()].groupby(GROUP_COLS).size()
forecast_horizon = int(future_counts.max()) if not future_counts.empty else 12
if forecast_horizon <= 0:
    forecast_horizon = 12

print(f'Горизонт прогнозу: {forecast_horizon} періодів')

['date', 'category_id', 'category_title', 'revenue', 'is_sale_prohibition', 'cos_month', 'sin_month', 'cos_quarter', 'sin_quarter', 'unique_brand_count']
        date  category_id      category_title       revenue  \
0 2019-01-01            1  Portable Computers  1.201956e+10   
1 2019-01-02            1  Portable Computers  1.334516e+10   
2 2019-01-03            1  Portable Computers  1.452505e+10   
3 2019-01-04            1  Portable Computers  9.928833e+09   
4 2019-01-05            1  Portable Computers  1.536200e+10   

   is_sale_prohibition  cos_month  sin_month  cos_quarter  sin_quarter  \
0                  NaN        NaN        NaN          NaN          NaN   
1                  NaN        NaN        NaN          NaN          NaN   
2                  NaN        NaN        NaN          NaN          NaN   
3                  NaN        NaN        NaN          NaN          NaN   
4                  NaN        NaN        NaN          NaN          NaN   

   unique_brand_count 

In [10]:
input_cols = ['date', *GROUP_COLS, TARGET_COLUMN, *REGRESSORS]
input_cols = list(dict.fromkeys(input_cols))
config = ForecastConfig(
    time_col='date',
    target_col=TARGET_COLUMN,
    group_cols=GROUP_COLS,
    freq='MS',
    forecast_horizon=forecast_horizon,
    seasonal_periods=12,
    min_history=24,
    lags=(1, 2, 3, 6, 12, 18, 24),
    rolling_windows=(3, 6, 12, 24),
    additional_regressors=REGRESSORS,
    random_search_iterations=10,
    n_splits=4,
    random_state=46,
)

preds, summaries = run_three_phase_forecast(df[input_cols].copy(), config)
preds = preds.rename(columns={
    'prediction': 'revenue_forecast',
    f'{TARGET_COLUMN}_holtwinters': 'revenue_baseline',
})
summary_report = pd.DataFrame({
    'group_key': [s.group_key[0] for s in summaries],
    'train_rows': [s.train_rows for s in summaries],
    'cv_mae': [s.best_score for s in summaries],
    'skipped_reason': [s.skipped_reason for s in summaries],
})
summary_report.head()

c:\Users\User\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\User\anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:918: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
c:\Users\User\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
c:\Users\User\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
c:\Users\User\anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\mo

,group_key,train_rows,cv_mae,skipped_reason
0,1,72,1.576877e+09,None
1,2,72,1.060737e+09,None
2,3,72,2.195055e+08,None
3,4,72,9.828227e+08,None
4,5,72,2.589368e+08,None


In [11]:
merge_cols = [*GROUP_COLS, 'date']
result_df = df.copy()
result_df['is_forecast_period'] = result_df[TARGET_COLUMN].isna()
result_df = result_df.merge(preds[merge_cols + ['revenue_forecast']], on=merge_cols, how='left')
result_df['revenue'] = result_df['revenue'].astype(float)
result_df['revenue'] = result_df['revenue'].fillna(result_df['revenue_forecast'])

output_columns = ['date', 'category_id', 'category_title', 'revenue']
final_output = result_df.loc[result_df['is_forecast_period'], output_columns]
final_output = final_output.groupby(['date', 'category_id', 'category_title'], as_index=False)['revenue'].sum()
final_output = final_output.sort_values(['category_id', 'date']).reset_index(drop=True)
final_output.to_csv(OUTPUT_PATH, index=False)

final_output.tail()

,date,category_id,category_title,revenue
343,2025-01-08,43,Communication Gadgets,1.721485e+09
344,2025-01-09,43,Communication Gadgets,1.448440e+09
345,2025-01-10,43,Communication Gadgets,1.729062e+09
346,2025-01-11,43,Communication Gadgets,1.730473e+09
347,2025-01-12,43,Communication Gadgets,1.728781e+09


In [ ]:
summary_report